# Lab 1 for EC3318/MN3101 Corporate Finance

**Contents**:

1. Starting your notebook
2. Loading stock price data
3. Computing stock returns
4. Visualising stock prices and returns

**Instructions**:

1. Read the notebook
2. Execute code cells as you encounter them
3. If you want help, ask GitHub Copilot to explain a concept or Python syntax. AI is optional; no AI subscription is needed.

## 1. Starting your notebook

In Codespaces, open this notebook and run the first code cell below with its play button (or Shift+Enter). Work through the cells in order.

If you see an import error:

1. Run **Terminal → Run Task → Setup lab environment** again.
2. Use **Select Kernel** at the top right and select **Corporate Finance Labs (.venv)**. If it is not listed, choose **Python Environments → .venv**.
3. Run the cell again.

A notebook combines short code cells, results and explanations. You can edit a cell and run it again to explore what changes. Setup details are in the [optional reference](docs/reference.md).

In [1]:
# Import essential data manipulation package
import pandas as pd
import numpy as np

**Ready to start**

Your environment is pre-configured and ready to use. Focus on your analysis - the technical setup is handled for you!


## 2. Loading stock price data

Now we'll work with real stock price data from our Excel file.


In [ ]:
# Load stock price data from Excel file
prices_raw = pd.read_excel(
    io=".xlsx",  # our Excel file
    sheet_name="Data download",  # Sheet in Excel file
    skiprows=7,  # Skip first 7 rows in sheet
)

prices_raw

**Why Excel data matters in finance**

Excel remains ubiquitous in financial practice. Investment banks, asset managers, and corporate finance teams routinely receive data in Excel format from Bloomberg, Refinitiv, and other financial data providers.

Learning to programmatically import Excel data is essential because:

- Manual copy-pasting introduces errors and is not reproducible
- Financial models require regular updates with new data
- Regulatory compliance often demands audit trails of data sources
- Large datasets exceed Excel's row limitations (1,048,576 rows)


In [ ]:
# Import datetime library for working with dates
from datetime import datetime, timedelta

# Convert Excel numeric date to a normal date
prices_raw["date"] = pd.to_datetime(
    prices_raw["date"],  # date column
    unit="D",  # in days
    origin="1899-12-30",  # the date that a number of zero represents
)

prices_raw

In [ ]:
# Shape of data
prices_raw.shape

In [ ]:
# Columns
prices_raw.columns.tolist()

In [ ]:
# Data types
prices_raw.dtypes

In [ ]:
# Date range
[
    prices_raw["date"].min(),  # earliest date in data
    prices_raw["date"].max(),  # latest date
]

In [ ]:
# Check for missing values
prices_raw.isnull().sum()

**Data quality checks: Foundation of financial analysis**

Before analyzing any financial data, rigorous quality checks are mandatory. Missing values, incorrect dates, and outliers can lead to:

- **Incorrect valuations** in DCF models
- **Flawed risk assessments** for portfolio management
- **Regulatory violations** in financial reporting
- **Trading losses** from bad signals

Professional practice requires documenting these checks for audit purposes and regulatory compliance (e.g., Basel III, Solvency II, IFRS 9).


In [ ]:
# Transform from wide format to tidy (long) format

# Melt the dataframe to convert from wide to long format
prices_tidy = pd.melt(
    prices_raw,  # our data
    id_vars=["date"],  # column that identifies rows in wide data
    var_name="symbol",  # new column for current variable names
    value_name="adj_close",  # new column for respective table values
)
prices_tidy

In [ ]:
# Sort by symbol and date for better organization
prices_tidy = (
    prices_tidy.get(["symbol", "date", "adj_close"])  # arrange columns differently
    .sort_values(["symbol", "date"])  # sort rows by symbol and date
    .reset_index(drop=True)  # maintain clean row identifiers
)

prices_tidy

**Tidy data: The foundation for scalable financial analysis**

Converting from "wide" to "long" format may seem technical, but it's crucial for financial modeling:

- **Portfolio analysis**: Compare hundreds of securities simultaneously
- **Risk management**: Calculate correlations across multiple assets
- **Performance attribution**: Analyze returns by sector, geography, or style
- **Regulatory reporting**: Format data for XBRL and other regulatory standards

This structure enables vectorized operations essential for high-frequency analysis and real-time risk monitoring.


## 3. Computing stock returns


**Stock returns: Comparing investments**

Simple returns measure the proportional change in value: $r_t=(P_t-P_{t-1})/P_{t-1}$. They let us compare stocks with different price levels. Returns may have more stable properties than prices, but their means and volatility can still change over time.

Simple returns **compound across time**; they are not additive. A +10% month followed by a −10% month gives $(1.10)(0.90)-1=-1\%$, not zero. For several periods, the cumulative return is $\prod_t(1+r_t)-1$.

For a portfolio over one period, returns are a weighted average of its assets' returns, using weights at the start of that period.

In [ ]:
# Calculate returns for each stock

returns = (
    prices_tidy.sort_values(["symbol", "date"])  # verify sorted rows
    .assign(
        ret=prices_tidy.groupby("symbol")["adj_close"].pct_change(fill_method=None)
    )  # calculate returns
    .reset_index(drop=True)  # clean row identifiers
)

# look at first five rows per stock
returns.groupby("symbol").head()

In [ ]:
# Drop unavailable returns, including the first row for each stock
returns = returns.dropna(subset=["ret"])

# look at first five rows per stock
returns.groupby("symbol").head()

**Missing returns**

The first price for each stock has no preceding price, so its return is missing. A missing price also prevents calculating the adjacent one-period returns. We use `fill_method=None` to avoid inventing prices by carrying the previous price forward, then drop unavailable returns. Check missing values before doing this: dropping observations can change the sample.

## 4. Visualising stock prices and returns


**Visualization in financial analysis: Beyond pretty charts**

Effective visualization is critical for:

- **Risk communication** to boards and regulators
- **Pattern recognition** in market anomalies and trading strategies
- **Performance reporting** to institutional investors
- **Stress testing** under different market scenarios

The plots we'll create mirror those used in professional investment management: price charts for performance tracking and return histograms for risk assessment.


In [ ]:
# Focus on Amazon stock
amazon_data = returns.query("symbol == 'AMZN'")  # filter for AMZN symbol
amazon_data

In [ ]:
# Summary statistics
pd.DataFrame(amazon_data["ret"].describe()).round(3).T

In [ ]:
# Yearly summary statistics
(amazon_data["ret"].groupby(amazon_data["date"].dt.year).describe().round(3))

**Summary statistics: The language of risk and return**

These statistics form the foundation of modern portfolio theory:

- **Mean**: Expected return (μ) - central to CAPM and asset pricing
- **Standard deviation**: Volatility (σ) - the classic risk measure
- **Min/Max**: Extreme values indicating tail risk
- **Count**: Sample size affecting statistical significance

Notice how these vary by year - this time-varying risk is why financial models use rolling windows and GARCH specifications.


In [ ]:
# Import plotnine for producing graphs
# See https://plotnine.org
from plotnine import *

In [ ]:
# Plot Amazon stock price
amazon_prices_figure = (
    ggplot(amazon_data, aes(x="date", y="adj_close"))
    + geom_line()
    + labs(x="", y="Adjusted close ($)", title="Amazon (AMZN) stock price")
    + theme(axis_text_x=element_text(rotation=45, hjust=1))
)
amazon_prices_figure

In [ ]:
# package for percentage scales
from mizani.formatters import percent_format

# Historical 5th percentile of monthly returns
quantile_05 = amazon_data["ret"].quantile(0.05)
var_95 = -quantile_05  # 95% monthly VaR as a loss fraction

# plot Amazon stock returns histogram with 95% VaR
amazon_returns_figure = (
    ggplot(amazon_data, aes(x="ret"))
    + geom_histogram(bins=100)
    + geom_vline(aes(xintercept=quantile_05), linetype="dashed")
    + labs(x="Monthly return", y="Number of months",
           title="Distribution of monthly Amazon stock returns",
           subtitle="Dashed line: historical 5th percentile of returns")
    + scale_x_continuous(labels=percent_format())
)
amazon_returns_figure

**Value-at-Risk (VaR): A historical loss threshold**

The dashed line is the **5th percentile of monthly returns**. The **95% monthly historical VaR**, expressed as a loss fraction, is the negative of this return quantile (`var_95`). For example, a −12% return quantile means a 12% VaR: about 5% of sample months have losses exceeding that threshold, subject to quantile interpolation.

This describes the historical sample, not a guaranteed future probability. VaR does not tell us how large losses beyond the threshold can be; Expected Shortfall summarises those tail losses.

In [ ]:
# Summary statistics for each stock through groupby-aggregation
summary_stats = (
    returns.groupby("symbol")["ret"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(3)
)
summary_stats

In [ ]:
# Annualized statistics
annualized_stats = (
    returns.groupby("symbol")["ret"]  # group by symbol and focus on returns
    .agg(["mean", "std"])  # get means and standard deviations
    .assign(
        ann_mean=lambda x: x["mean"] * 12,  # annualise mean return
        ann_vol=lambda x: x["std"] * np.sqrt(12),  # annualise volatility
    )
    .get(["ann_mean", "ann_vol"])
    .round(3)
)
annualized_stats

**Annualisation: A reporting convention**

We report the **arithmetic annualised mean** as $12\bar r_{\text{monthly}}$ and annualised volatility as $\sqrt{12}s_{\text{monthly}}$. These are the conventions used in Lab 2's mean–variance analysis.

The mean is a scaled monthly estimate, **not a compounded annual return**. Realised returns over 12 months compound as $\prod_{t=1}^{12}(1+r_t)-1$. Under independent identically distributed monthly returns, the expected compounded annual return is $(1+E[r])^{12}-1$.

Square-root-of-time volatility scaling assumes no serial covariance and stable monthly variance. It describes the sum of monthly returns; it is an approximation for compounded annual simple-return volatility.

In [ ]:
# more formatting functions for plotnine
from mizani.breaks import date_breaks
from mizani.formatters import date_format

# plot all stock prices at once
prices_figure = (
    ggplot(returns, aes(y="adj_close", x="date", color="symbol"))
    + geom_line()
    + scale_x_datetime(breaks=date_breaks(width="1 year"), labels=date_format("%Y"))
    + labs(x="Date", y="Adjusted close ($)", color="", title="Adjusted stock prices")
    + theme(legend_position="right")
)
prices_figure.show()

In [ ]:
# Plot return series for all stocks
returns_figure = (
    ggplot(returns, aes(y="ret", x="date", color="symbol"))
    + geom_line()
    + scale_x_datetime(breaks=date_breaks(width="1 year"), labels=date_format("%Y"))
    + labs(x="Date", y="Monthly return", color="", title="Monthly returns by stock")
    + scale_y_continuous(labels=percent_format())
    + theme(legend_position="right")
)
returns_figure.show()

**Cross-sectional analysis: The foundation of portfolio theory**

Comparing multiple securities simultaneously reveals:

- **Correlation patterns**: Essential for diversification benefits
- **Relative performance**: Identifying alpha generation opportunities
- **Sector dynamics**: Understanding systematic vs. idiosyncratic risk
- **Mean reversion**: Potential arbitrage opportunities

This multi-asset visualization is how portfolio managers monitor:

- Factor exposures (growth vs. value, large vs. small cap)
- Geographic allocation effects
- Currency hedging decisions
- Rebalancing triggers

The varying correlations visible in these charts drive the benefits of diversification central to modern portfolio theory.
